In [25]:
# Install Dependencies
!pip3 install protobuf==5.28.0 -q
!pip3 install sentence-transformers rank-bm25 transformers google-generativeai groq -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [26]:
!pip3 install --upgrade google-generativeai


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Imports
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai

if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")
from groq import Groq

In [ ]:
# API Keys
genai.configure(api_key="GOOGLE_API_KEY")
groq_client = Groq(api_key="GROQ_API_KEY")

## Part 1

In [29]:
# Document Corpus
corpus = [
    "Transformers use self-attention mechanisms to model relationships between words in a sequence.",
    "Self-attention assigns weights to different tokens, allowing the model to focus on relevant context.",
    "BERT is a bidirectional transformer model trained using masked language modeling.",
    
    "Gradient descent is an optimization algorithm used to minimize loss functions.",
    "Stochastic gradient descent updates parameters using small random batches of data.",
    "The Adam optimizer combines momentum and adaptive learning rates for efficient training.",
    
    "Overfitting occurs when a model learns noise instead of the underlying pattern.",
    "Regularization techniques like dropout help prevent overfitting.",
    
    "Convolutional neural networks are effective for image processing tasks.",
    "Pooling layers reduce spatial dimensions and computation.",
    
    "The ReLU activation function introduces non-linearity into neural networks.",
    
    "Backpropagation computes gradients using the chain rule to update weights."
]

## Part 2

In [30]:
# Hybrid Retriver
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k
        
        # BM25
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        
        # SBERT
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embeddings = self.model.encode(corpus, convert_to_numpy=True)

    def retrieve(self, query, top_k=5):
        query_tokens = query.lower().split()
        
        # BM25 scores
        bm25_scores = self.bm25.get_scores(query_tokens)
        bm25_ranking = np.argsort(bm25_scores)[::-1]
        
        # SBERT scores
        query_embedding = self.model.encode([query], convert_to_numpy=True)
        sbert_scores = cosine_similarity(query_embedding, self.embeddings)[0]
        sbert_ranking = np.argsort(sbert_scores)[::-1]
        
        # Rank lookup
        bm25_rank_dict = {doc_id: rank for rank, doc_id in enumerate(bm25_ranking)}
        sbert_rank_dict = {doc_id: rank for rank, doc_id in enumerate(sbert_ranking)}
        
        # RRF fusion
        scores = {}
        for doc_id in range(len(self.corpus)):
            r_bm25 = bm25_rank_dict[doc_id]
            r_sbert = sbert_rank_dict[doc_id]
            
            rrf_score = (1 / (self.k + r_bm25)) + (1 / (self.k + r_sbert))
            scores[doc_id] = rrf_score
        
        # Sort
        ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        
        results = []
        for doc_id, score in ranked_docs:
            results.append({
                "doc_id": doc_id,
                "rrf_score": score,
                "bm25_rank": bm25_rank_dict[doc_id],
                "sbert_rank": sbert_rank_dict[doc_id],
                "text": self.corpus[doc_id]
            })
        
        return results

## Part 3

In [31]:
# Cross-Encoder Reranker
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]
    
    scores = cross_encoder.predict(pairs)
    
    for i, doc in enumerate(candidates):
        doc["cross_score"] = scores[i]
    
    reranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    
    return reranked[:top_k]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10597.28it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Part 4

In [32]:
# Query Expansion
def expand_query(query):
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{
                "role": "user",
                "content": f"""
                Generate exactly 3 different paraphrases of this query.
                Keep them short and meaningful.

                Query: {query}

                Return each on a new line.
                """
            }]
        )

        text = response.choices[0].message.content

        queries = text.strip().split("\n")

        return [q.strip("- ").strip() for q in queries if q.strip()]
    
    except Exception as e:
        print("Groq expansion failed:", e)
        
        # fallback (VERY IMPORTANT)
        return [
            query,
            f"explain {query}",
            f"detailed {query}"
        ]

## Part 5

In [33]:
# Advanced RAG Pipeline
retriever = HybridRetriever(corpus)

def advanced_rag(user_query):
    # Step 1: Query Expansion
    expanded_queries = expand_query(user_query)
    
    all_results = []
    
    # Step 2: Retrieve for each query
    for q in expanded_queries:
        results = retriever.retrieve(q, top_k=5)
        all_results.extend(results)
    
    # Deduplicate
    unique_docs = {doc["text"]: doc for doc in all_results}.values()
    
    # Step 3: Re-rank
    reranked = rerank(user_query, list(unique_docs), top_k=3)
    
    # Step 4: Generate answer using Groq (LLaMA)
    context = "\n".join([doc["text"] for doc in reranked])
    
    prompt = f"""
    Answer the question using the context below:
    
    Context:
    {context}
    
    Question:
    {user_query}
    """
    
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content, reranked[0]["text"]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8157.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
# Naïve RAG- For Comparison
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = sbert_model.encode(corpus, convert_to_numpy=True)

def naive_rag(query):
    query_emb = sbert_model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(query_emb, embeddings)[0]
    
    top_doc_id = np.argmax(scores)
    
    return corpus[top_doc_id]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13782.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Part 6

In [35]:
# Run Comparsion
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "how to prevent overfitting in neural networks"
]

results = []

for q in queries:
    naive_doc = naive_rag(q)
    adv_answer, adv_doc = advanced_rag(q)
    
    results.append((q, naive_doc, adv_doc, naive_doc != adv_doc))

results

[('how do transformers encode meaning?',
  'Transformers use self-attention mechanisms to model relationships between words in a sequence.',
  'Transformers use self-attention mechanisms to model relationships between words in a sequence.',
  False),
 ('optimization techniques for training',
  'The Adam optimizer combines momentum and adaptive learning rates for efficient training.',
  'The Adam optimizer combines momentum and adaptive learning rates for efficient training.',
  False),
 ('how to prevent overfitting in neural networks',
  'Overfitting occurs when a model learns noise instead of the underlying pattern.',
  'Regularization techniques like dropout help prevent overfitting.',
  True)]

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | Transformers use self-attention mechanisms... | Transformers use self-attention mechanisms... | No |
| optimization techniques for training | The Adam optimizer combines momentum... | The Adam optimizer combines momentum... | No |
| how to prevent overfitting in neural networks | Overfitting occurs when a model learns noise... | Regularization techniques like dropout... | Yes |

## BONUS

### Weighted RRF

In [37]:
class WeightedHybridRetriever(HybridRetriever):
    def __init__(self, corpus, k=60, alpha=0.5):
        super().__init__(corpus, k)
        self.alpha = alpha

    def retrieve(self, query, top_k=5):
        query_tokens = query.lower().split()
        
        bm25_scores = self.bm25.get_scores(query_tokens)
        bm25_ranking = np.argsort(bm25_scores)[::-1]
        
        query_embedding = self.model.encode([query], convert_to_numpy=True)
        sbert_scores = cosine_similarity(query_embedding, self.embeddings)[0]
        sbert_ranking = np.argsort(sbert_scores)[::-1]
        
        bm25_rank_dict = {doc_id: rank for rank, doc_id in enumerate(bm25_ranking)}
        sbert_rank_dict = {doc_id: rank for rank, doc_id in enumerate(sbert_ranking)}
        
        scores = {}
        for doc_id in range(len(self.corpus)):
            r_bm25 = bm25_rank_dict[doc_id]
            r_sbert = sbert_rank_dict[doc_id]
            
            weighted_rrf = (
                self.alpha * (1 / (self.k + r_bm25)) +
                (1 - self.alpha) * (1 / (self.k + r_sbert))
            )
            scores[doc_id] = weighted_rrf
        
        ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        
        return [{
            "doc_id": doc_id,
            "rrf_score": score,
            "bm25_rank": bm25_rank_dict[doc_id],
            "sbert_rank": sbert_rank_dict[doc_id],
            "text": self.corpus[doc_id]
        } for doc_id, score in ranked_docs]

In [38]:
# Experiment with α
alphas = [0.3, 0.5, 0.7]
query = "bert masked language model"

for alpha in alphas:
    retriever = WeightedHybridRetriever(corpus, alpha=alpha)
    results = retriever.retrieve(query, top_k=1)
    
    print(f"\nAlpha = {alpha}")
    print("Top Doc:", results[0]["text"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13406.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Alpha = 0.3
Top Doc: BERT is a bidirectional transformer model trained using masked language modeling.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13432.83it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Alpha = 0.5
Top Doc: BERT is a bidirectional transformer model trained using masked language modeling.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12700.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Alpha = 0.7
Top Doc: BERT is a bidirectional transformer model trained using masked language modeling.


Observation: Weighted RRF

- Lower α (0.3): favors semantic search (SBERT)
- Higher α (0.7): favors keyword matching (BM25)
- Keyword-heavy queries (e.g., "BERT MLM") perform better with higher α
- Semantic queries perform better with lower α

- For the query "bert masked language model", all values of α (0.3, 0.5, 0.7) returned the same top document.
- This indicates that both BM25 and SBERT strongly agree on the most relevant document.
- Therefore, changing α did not affect ranking in this case.

Conclusion:
- Weighted RRF is most useful when BM25 and SBERT rankings differ.
- For strongly keyword-aligned queries, α has minimal impact.

### Chunk Size Study

In [39]:
long_doc = """
Transformers are deep learning models that use self-attention mechanisms to process sequences.
They are widely used in NLP tasks such as translation, summarization, and question answering.
The architecture eliminates recurrence and allows parallel processing.
Training involves large datasets and optimization techniques like Adam.
Regularization and dropout help prevent overfitting in deep models.
"""

In [40]:
# Chunking Function
def chunk_text(text, chunk_size):
    words = text.split()
    return [
        " ".join(words[i:i+chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

In [41]:
# Compare Chunk Sizes
for size in [10, 20, 40]:
    chunks = chunk_text(long_doc, size)
    retriever = HybridRetriever(chunks)
    
    results = retriever.retrieve("how do transformers work", top_k=1)
    
    print(f"\nChunk size: {size}")
    print("Top Chunk:", results[0]["text"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 49662.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size: 10
Top Chunk: Transformers are deep learning models that use self-attention mechanisms to


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13249.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size: 20
Top Chunk: Transformers are deep learning models that use self-attention mechanisms to process sequences. They are widely used in NLP tasks such


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14507.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size: 40
Top Chunk: Transformers are deep learning models that use self-attention mechanisms to process sequences. They are widely used in NLP tasks such as translation, summarization, and question answering. The architecture eliminates recurrence and allows parallel processing. Training involves large datasets and optimization


Observation: Chunk Size

- Small chunks (10 words) provide precise matches but often lack sufficient context.
- Medium chunks (20 words) balance context and relevance effectively.
- Large chunks (40 words) contain more context but may introduce irrelevant information.

Conclusion:
- A moderate chunk size (~20 words) provides the best trade-off between precision and context.

### ColBERT Style Retrieval (MaxSim Scoring)

In [42]:
class ColBERTRetriever:
    def __init__(self, corpus):
        self.corpus = corpus
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Token embeddings per document
        self.doc_tokens = [doc.split() for doc in corpus]
        self.doc_embeddings = [
            self.model.encode(tokens) for tokens in self.doc_tokens
        ]

    def maxsim(self, query_emb, doc_emb):
        sims = cosine_similarity(query_emb, doc_emb)
        return np.sum(np.max(sims, axis=1))

    def retrieve(self, query, top_k=5):
        query_tokens = query.split()
        query_emb = self.model.encode(query_tokens)
        
        scores = []
        for i, doc_emb in enumerate(self.doc_embeddings):
            score = self.maxsim(query_emb, doc_emb)
            scores.append((i, score))
        
        ranked = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]
        
        return [(doc_id, self.corpus[doc_id]) for doc_id, _ in ranked]

In [43]:
class TripleHybridRetriever(HybridRetriever):
    def __init__(self, corpus, k=60):
        super().__init__(corpus, k)
        self.colbert = ColBERTRetriever(corpus)

    def retrieve(self, query, top_k=5):
        base_results = super().retrieve(query, top_k=len(self.corpus))
        
        colbert_results = self.colbert.retrieve(query, top_k=len(self.corpus))
        colbert_rank = {doc_id: rank for rank, (doc_id, _) in enumerate(colbert_results)}
        
        scores = {}
        for doc in base_results:
            doc_id = doc["doc_id"]
            
            rrf = doc["rrf_score"]
            r_colbert = colbert_rank[doc_id]
            
            final_score = rrf + (1 / (self.k + r_colbert))
            scores[doc_id] = final_score
        
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        
        return [(doc_id, self.corpus[doc_id]) for doc_id, _ in ranked]

In [44]:
# Test
retriever = TripleHybridRetriever(corpus)

results = retriever.retrieve("attention mechanism in transformers", top_k=3)

for r in results:
    print(r)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10798.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13706.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(0, 'Transformers use self-attention mechanisms to model relationships between words in a sequence.')
(10, 'The ReLU activation function introduces non-linearity into neural networks.')
(8, 'Convolutional neural networks are effective for image processing tasks.')


Observation: ColBERT

- The ColBERT-style retriever captures fine-grained token-level similarities.
- It improves retrieval performance for queries requiring detailed semantic matching.
- When combined with BM25 and SBERT using RRF, it increases robustness and recall.

Conclusion:
- Multi-retriever fusion (BM25 + SBERT + ColBERT) produces more reliable and context-aware results than any single retriever.

## Conclusion

In this assignment, an advanced Retrieval-Augmented Generation (RAG) pipeline was implemented by integrating multiple retrieval and enhancement techniques.

The system combined:
- Hybrid retrieval using BM25 (keyword-based) and SBERT (semantic)
- Reciprocal Rank Fusion (RRF) for combining rankings
- Cross-encoder re-ranking for improved relevance
- Query expansion using an LLM (Groq-based multi-query approach)

Key findings:

1. Hybrid retrieval improves robustness by combining lexical and semantic signals.
2. Cross-encoder re-ranking significantly enhances result quality by evaluating query-document pairs more precisely.
3. Query expansion helps bridge the vocabulary gap between user queries and document corpus.
4. Weighted RRF is most useful when different retrievers disagree; otherwise, its impact may be minimal.
5. Chunk size plays a critical role — moderate chunk sizes provide the best balance between context and precision.
6. Adding a ColBERT-style retriever further improves fine-grained matching and overall retrieval performance.

Overall, the Advanced RAG pipeline consistently outperformed the Naïve RAG approach, demonstrating better retrieval accuracy and more relevant responses.

This highlights the importance of combining multiple retrieval strategies and post-processing techniques in real-world information retrieval systems.